# FPU — Learned Potential Evolution

Plot the evolution of the learned potential $V_\theta(u(t))$ over time
for 10 test trajectories.

- **Learned potential** (`s_onsagernet` model): $V_\theta(u) = V_0 + V_1 + V_2$

Each colored line is one test trajectory. A monotonically decreasing $V_\theta$
would indicate the model has learned a valid Lyapunov function for the dynamics.

In [ ]:
from pathlib import Path

import rootutils
import torch

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from notebooks.potential_evolution.helpers import (  # noqa: E402
    compute_V_learned,
    load_model_for_inference,
    load_test_data,
    plot_V_change,
    plot_V_evolution,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT: {ROOT}")
print(f"Device: {device}")

In [ ]:
RUN_DIR = ROOT / "logs/official/runs/ckdv/s_onsagernet"
print("Loading 's_onsagernet' (learned potential) ...")
model = load_model_for_inference(RUN_DIR, root=ROOT, device=device)
potential = model.dynamics.potential
print(f"Potential type: {type(potential).__name__}")

In [ ]:
test_data, t_coord, x_coord = load_test_data(str(ROOT / "data/fput/*.hdf5"))
N_test, T, n_vars, Nx = test_data.shape
print(f"Test set: {N_test} trajectories  |  T={T}, n_vars={n_vars}, Nx={Nx}")

In [ ]:
V_learned = compute_V_learned(potential, test_data, device)
print(f"V_learned shape: {V_learned.shape}")
print(f"V_learned range: [{V_learned.min():.3e}, {V_learned.max():.3e}]")

In [ ]:
plot_V_evolution(V_learned, ROOT / "figs/potential_evolution/fpu_potential_evolution.pdf")

In [ ]:
plot_V_change(V_learned, ROOT / "figs/potential_evolution/fpu_potential_change.pdf")